# Notebook 29 — scorer-vs-human examples, for review

Supplementary review material. **No experiment is run here.** Every item,
verdict and label already exists; this notebook only puts them next to the
handwritten page and writes captioned PNG sheets that open directly in Drive.

**What is being classified.** The *prediction* is the automatic scorer's
verdict (`strict_v1`); the *truth* is a human reading of the page. So
**TP/FP/FN/TN describe the SCORER, not the model.**

| group | meaning | n audited |
|---|---|---|
| **TP** | scorer said CORRECT, human agrees | 95 |
| **FP** | scorer said CORRECT, verdict UNEARNED (false pass) | 15 |
| **FN** | scorer said WRONG, model was actually correct (false fail) | 33 |
| **TN** | scorer said WRONG, human agrees the model was wrong | 20 |
| **INDETERMINATE** | scorer said WRONG and the verdict was unearned | 70 |
| **NEEDS_VISUAL** | scorer said CORRECT but the coder could not read the page | 1 |

**Six groups, not four, and the extra two are not padding.** The audit's
`extraction_issue` label means *the verdict was not earned*, which is not the
same as *the model was wrong*. On a pass that is a false pass; on a fail it
leaves the model's answer undecided, so it is neither TN nor FN. Folding those
71 items into TN would inflate the scorer's apparent accuracy on the largest
single group in the audit, and they are the project's main finding.
`NEEDS_VISUAL` holds the one item the coder could not decide *while the scorer
passed it*: weaker than a false pass, and worth a second look rather than a
quiet drop.

**Every sheet title states a scorer verdict and every item on that sheet has
it**, asserted before anything is written. Item 5 previously appeared under a
heading reading "scorer said WRONG" while its own caption read
`v1=CORRECT`.

Specific cases requested for the sheets are forced in and appear first in
their group: **180, 289, 291** (false passes), **144** (a false fail where the
model answered in prose and the truth collapsed to `sympy:2`), **132, 147**
(clean passes), **299, 273** (unearned fails). The rest of each group is a
seeded draw that round-robins over the human label, so a group built from
several mechanisms shows several rather than six copies of one.

**TP exemplars must show the answer.** Item 77 is a genuine true pass whose
span is the single letter `m` while the page holds two coefficient tables; as
an illustration it is indistinguishable from the collapse cases on the FP
sheet. It is excluded by name, and TP selection now prefers spans of at least
four characters so none of its eight siblings replaces it. This is a choice of
*example*, not a recategorisation: those items are still true passes in the
counts above.

No GPU. Reads the dataset and one results CSV; changes no scorer rule and
computes no accuracy.

In [ ]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and an existing results CSV, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. This cost 43/300 items once,
# and here it would change the very labels burned into the captions.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.audit_diagnostics
import pilot.canonicalize
import pilot.data
import pilot.dataset_profile
import pilot.plotting
import pilot.rescore
import pilot.strict_v2

print(f"pilot imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert hasattr(pilot.dataset_profile, "confusion_examples"), (
    "the repo clone predates notebook 29's library code -- push "
    "pilot/dataset_profile.py and re-run this cell. A green local dry run "
    "does NOT cover this: the dry run uses the working tree, Colab uses the "
    "remote.")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label would fall back to the "
    "plain-text tier, so the `label M`/`label T` lines on these sheets would "
    "show something the frozen scorer never saw.")
print("SymPy LaTeX parser OK")

In [ ]:
import pandas as pd

RUN_CSV = "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv"
SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5
PER_PAGE = 6

run = pd.read_csv(f"{RESULTS_DIR}/{RUN_CSV}")
assert len(run) == 300, f"expected 300 rows, got {len(run)}"
print(f"run: {len(run)} rows, model={run['model_id'].unique().tolist()}")

sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)

# load_fermat_balanced SHUFFLES its final selection, so index alignment is an
# assumption to verify, not one to make. A caption under the wrong handwritten
# page would be worse than no sheet at all, and this is review material a
# professor will read at face value.
assert len(sample) == len(run), f"{len(sample)} items vs {len(run)} rows"
bad = [i for i in range(len(run))
       if sample[i]["orig_q"].strip() != str(run.iloc[i]["orig_q"]).strip()]
assert not bad, (
    f"{len(bad)} rows where the rebuilt sample's question does not match the "
    f"CSV's (first: {bad[:5]}). Images would be attached to the wrong rows.")
print("sample order matches the CSV on all 300 rows -- images are index-aligned")

v1 = pilot.rescore.rescore_run(run, "strict_v1", progress=True)["transcription_correct"].astype(bool)
v2s = pilot.strict_v2.rescore_v2(run, progress=True)

# Human labels, deduplicated by the audit sets' own precedence (later, more
# specific pass wins) -- the same rule audit_diagnostics uses.
audits = pilot.audit_diagnostics.load_audit_sets("repo/reference/audit")
frames = []
for name in pilot.audit_diagnostics.SET_PRECEDENCE:
    d = audits[name].copy()
    d["set"] = name
    d["_p"] = list(pilot.audit_diagnostics.SET_PRECEDENCE).index(name)
    frames.append(d)
audit = pd.concat(frames).sort_values("_p")
audit = audit[~audit.index.duplicated(keep="first")]
print(f"audited items: {len(audit)} of 300")

population = pd.DataFrame({
    "item_id": audit.index,
    "category": [pilot.dataset_profile.confusion_category(
        v1.loc[i], audit.loc[i, "final_label"]) for i in audit.index]})
print(population["category"].value_counts().to_string())

In [ ]:
# The examples. Requested cases are forced in and appear first in their group.
PREFER = {
    "FP": [180, 289, 291],      # false passes discussed already
    "FN": [144],                # prose answer vs a collapsed `sympy:2` truth
    "TP": [132, 147],           # clean passes
    "INDETERMINATE": [299, 273],  # unearned FAILS, deliberately not TN
}
N_PER_CATEGORY = 6

# Rejected by name on review. Item 77 IS a genuine true pass -- the coder's own
# note reads "tiny span but a valid table cell" -- but its span is the single
# letter `m` while the page holds two coefficient tables, so as an EXEMPLAR it
# shows nothing and looks like the collapse cases on the FP sheet. The general
# rule below (TP prefers spans of at least MIN_EXEMPLAR_SPAN characters) is
# what stops one of its eight siblings taking its place.
EXCLUDE = [77]

examples = pilot.dataset_profile.confusion_examples(
    run, v1, v2s, audit, n_per_category=N_PER_CATEGORY, prefer=PREFER,
    per_page=PER_PAGE, exclude=EXCLUDE)

tp_spans = [(int(r["item_id"]), len(" ".join(str(r["span_m_disp"]).split())))
            for _, r in examples[examples["category"] == "TP"].iterrows()]
assert all(n >= pilot.dataset_profile.MIN_EXEMPLAR_SPAN for _, n in tp_spans), (
    f"a TP exemplar has a span too short to show the answer: {tp_spans}")
print(f"TP span lengths: {sorted(n for _, n in tp_spans)}")

missing = [i for v in PREFER.values() for i in v
           if i not in set(examples["item_id"])]
assert not missing, (
    f"requested items absent from the sheets: {missing}. They may have fallen "
    "into a different confusion cell than expected -- check before rendering.")

# Every sheet title states a scorer verdict; every item on it must have that
# verdict. Checked BEFORE rendering, because a caption contradicting its own
# heading is read as a contradiction in the data by someone without context.
print(pilot.dataset_profile.assert_confusion_groups_match_titles(examples))
print(f"{len(examples)} examples across {examples['category'].nunique()} groups")
print(pd.crosstab(examples["category"], examples["human_label"]).to_string())

In [ ]:
# Captioned contact sheets. PNGs are what makes a folder reviewable IN DRIVE:
# bare per-item images are a wall of uncaptioned thumbnails and HTML does not
# render in Drive's preview.
import matplotlib.pyplot as plt

SHEET_DIR = f"{PROJECT_DIR}/figures/confusion_examples_for_professor"
os.makedirs(SHEET_DIR, exist_ok=True)

TITLES = {
    "TP": "TRUE PASS - scorer said CORRECT and the human agrees",
    "FP": "FALSE PASS - scorer said CORRECT, human found the verdict UNEARNED",
    "FN": "FALSE FAIL - scorer said WRONG, human found the model CORRECT",
    "TN": "TRUE FAIL - scorer said WRONG and the human agrees",
    "INDETERMINATE": ("UNEARNED FAIL - scorer said WRONG for a reason that "
                      "does not hold; the model's answer is UNKNOWN"),
    "NEEDS_VISUAL": ("NEEDS A VISUAL CHECK - scorer said CORRECT but the page "
                     "could not be read confidently; correctness UNKNOWN"),
}

for cat in pilot.dataset_profile.CONFUSION_ORDER:
    sub = examples[examples["category"] == cat]
    if not len(sub):
        continue
    figs = pilot.plotting.contact_sheet(
        [sample[int(i)]["image"] for i in sub["item_id"]],
        [pilot.dataset_profile.confusion_caption(r) for _, r in sub.iterrows()],
        ncols=3, per_page=PER_PAGE, cell_height=6.2, caption_fontsize=5.5,
        title=TITLES[cat], footer=pilot.dataset_profile.CONFUSION_LEGEND)
    for page, fig in enumerate(figs, 1):
        fig.savefig(f"{SHEET_DIR}/confusion_{cat}_p{page}.png", dpi=150,
                    facecolor=fig.get_facecolor())
        plt.close(fig)
    print(f"{cat:14s} {len(sub)} items -> {len(figs)} page(s)")

# Every sheet the manifest names must exist, or a reader follows the CSV to a
# page that was never written.
missing = sorted({f for f in examples["contact_sheet_file"]
                  if not os.path.exists(f"{SHEET_DIR}/{f}")})
assert not missing, f"manifest names sheets that were not written: {missing}"
print(f"\nall {examples['contact_sheet_file'].nunique()} sheet files exist")

In [ ]:
# Manifest + README, written next to the sheets so the folder stands alone.
examples.to_csv(f"{SHEET_DIR}/manifest.csv", index=False)
readme = pilot.dataset_profile.write_confusion_readme(
    f"{SHEET_DIR}/README.md", examples, population=population)
print(f"manifest -> {SHEET_DIR}/manifest.csv  ({len(examples)} rows)")
print(f"README   -> {SHEET_DIR}/README.md")
print()
print(readme[:1800])
print("\n" + "=" * 70)
print("Open: My Drive > uncertainty-math-vlm > figures >")
print("      confusion_examples_for_professor")
print("""
Share that folder. The PNGs preview directly in Drive; README.md explains the
five groups and how to read a tile.

These sheets are ILLUSTRATIVE, not a measurement: items were chosen to show
each group and each mechanism, so nothing on them should be counted.
""")